# 04 — Chronological Splits, Occurrence-Level Baselines, and Next-Occurrence Targets

This notebook prepares the canonical leakage-safe artifacts used by downstream TraceAdvisor modeling.

## Responsibilities

1. Load the execution-level artifact produced by Notebook 03.
2. Validate only the N03 fields that N04 truly requires.
3. Create one deterministic chronological split targeting 70/15/15 while preserving complete `execution_start_time_us` groups.
4. Collapse simultaneous execution rows into one **recurrence occurrence** identified by `recurrence_key + execution_start_time_us`.
5. Build **past-only, completed-occurrence** history statistics for CPU, memory, and runtime.
6. Construct **next distinct occurrence** CPU/memory targets inside each frozen split so labels never cross split boundaries.
7. Generate Request, Mean, P90, P95, and P99 baselines using occurrence-level history.
8. Evaluate every baseline against the **same next-occurrence targets on the same occurrence population**.
9. Select `min_runs` and the historical baseline policy using validation only.
10. Materialize the final test artifact without scoring test performance.
11. Save both canonical occurrence-level artifacts and compatibility execution-level artifacts.

### Canonical modeling/evaluation unit

The canonical unit in this notebook is an **occurrence**, not an execution row. Multiple execution rows that share the same `recurrence_key` and `execution_start_time_us` are one occurrence.

### Prediction-time convention

A row represents the state available at the **start of the current occurrence**. Historical statistics include only earlier occurrences that have fully completed by that start time. The target is the peak CPU/memory of the next distinct occurrence in the same recurrence and split. The Request baseline carries forward the current occurrence request as the simple reference recommendation for that next occurrence.


In [38]:
from pathlib import Path
import heapq
import json
import sys

import numpy as np
import pandas as pd


In [39]:
# Locate repository root.
REPO_ROOT = Path.cwd()

while (
    REPO_ROOT.name != "masters-project"
    and REPO_ROOT.parent != REPO_ROOT
):
    REPO_ROOT = REPO_ROOT.parent

if REPO_ROOT.name != "masters-project":
    raise RuntimeError(
        "Could not locate masters-project repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)


Repository root: /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project


In [40]:
from src.features.split_utils import chronological_split

DATA_DIR = REPO_ROOT / "data" / "processed" / "google_subset_v1"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_FEATURES_PATH = DATA_DIR / "job_metrics_v0.parquet"

# Compatibility execution-row artifacts (same filenames used by earlier N04 versions).
TRAIN_FEATURES_PATH = DATA_DIR / "train_features.parquet"
VAL_FEATURES_PATH = DATA_DIR / "val_features.parquet"
TEST_FEATURES_PATH = DATA_DIR / "test_features.parquet"

# Canonical occurrence-level artifacts for N05/N06 modeling and evaluation.
TRAIN_OCCURRENCES_PATH = DATA_DIR / "train_occurrences.parquet"
VAL_OCCURRENCES_PATH = DATA_DIR / "val_occurrences.parquet"
TEST_OCCURRENCES_PATH = DATA_DIR / "test_occurrences.parquet"

VALIDATION_COMPARISON_PATH = DATA_DIR / "baseline_validation_comparison.csv"
METADATA_PATH = DATA_DIR / "metadata.json"

MIN_RUNS_CANDIDATES = [1, 3, 5, 10]

print("Data directory:", DATA_DIR)
print("Notebook 03 input:", PROFILE_FEATURES_PATH)
print("Candidate min_runs values:", MIN_RUNS_CANDIDATES)


Data directory: /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1
Notebook 03 input: /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/job_metrics_v0.parquet
Candidate min_runs values: [1, 3, 5, 10]


## Canonical N03 → N04 Contract

N04 only requires execution identity/timing, request fields, and observed peak resource outcomes from N03. Older N03 historical-profile columns may still be present, but N04 does **not** use their execution-row counts/statistics for baseline eligibility because `min_runs` must count completed **occurrences**, not execution rows.


In [41]:
BASE_REQUIRED_COLUMNS = [
    "execution_id",
    "recurrence_key",
    "execution_start_time_us",
    "execution_end_time_us",
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

OPTIONAL_N03_HISTORY_COLUMNS = [
    "hist_peak_cpu_count",
    "hist_peak_memory_count",
    "hist_peak_cpu_mean",
    "hist_peak_memory_mean",
    "hist_peak_cpu_p90",
    "hist_peak_memory_p90",
    "hist_peak_cpu_p95",
    "hist_peak_memory_p95",
    "hist_peak_cpu_p99",
    "hist_peak_memory_p99",
]


def validate_base_schema(df: pd.DataFrame) -> None:
    missing_columns = [
        column for column in BASE_REQUIRED_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required canonical columns: "
            f"{missing_columns}"
        )

    if df["execution_id"].isna().any():
        raise ValueError("execution_id contains missing values.")

    if df["execution_id"].duplicated().any():
        raise ValueError("execution_id must be unique.")

    if df["recurrence_key"].isna().any():
        raise ValueError("recurrence_key contains missing values.")

    if df["execution_start_time_us"].isna().any():
        raise ValueError("execution_start_time_us contains missing values.")

    if df["execution_end_time_us"].isna().any():
        raise ValueError("execution_end_time_us contains missing values.")

    if (
        df["execution_end_time_us"]
        < df["execution_start_time_us"]
    ).any():
        raise ValueError(
            "Found execution_end_time_us earlier than execution_start_time_us."
        )

    print("Canonical base schema validation passed.")


## Load Notebook 03 Output


In [42]:
if not PROFILE_FEATURES_PATH.exists():
    raise FileNotFoundError(
        "Notebook 03 output was not found. "
        f"Expected: {PROFILE_FEATURES_PATH}. "
        "Run 03_buildProfilesAndMetrics.ipynb first."
    )

execution_features = pd.read_parquet(PROFILE_FEATURES_PATH)

print("Loaded execution features:", execution_features.shape)
print("Unique execution IDs:", execution_features["execution_id"].nunique())
display(execution_features.head())


Loaded execution features: (253585, 50)
Unique execution IDs: 253585


,execution_id,collection_id,instance_index,candidate_instance_key,episode_number,execution_start_time_us,execution_end_time_us,observation_count,machine_count,requested_cpu_ncu,...,hist_peak_memory_p90,hist_peak_memory_p95,hist_peak_memory_p99,hist_runtime_count,hist_runtime_mean,hist_runtime_std,hist_runtime_p50,hist_runtime_p90,hist_runtime_p95,hist_runtime_p99
0,83212368472||1008||episode_1,83212368472,1008,83212368472||1008,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,83212368472||1042||episode_1,83212368472,1042,83212368472||1042,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
2,83212368472||1056||episode_1,83212368472,1056,83212368472||1056,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
3,83212368472||1096||episode_1,83212368472,1096,83212368472||1096,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
4,83212368472||1099||episode_1,83212368472,1099,83212368472||1099,1,2097300000000,2097600000000,1,1,0.031433,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
validate_base_schema(execution_features)


Canonical base schema validation passed.


## Missing-Value and Upstream-History Audit

No global `dropna()` is applied. CPU and memory can have different missingness. Older N03 history columns are shown only as an audit; the corrected occurrence-level baselines below do not depend on them.


In [44]:
important_columns = [
    "requested_cpu_ncu",
    "requested_memory_normalized",
    "peak_cpu_ncu",
    "peak_memory_normalized",
]

present_optional_history = [
    column for column in OPTIONAL_N03_HISTORY_COLUMNS
    if column in execution_features.columns
]

missing_summary = (
    execution_features[important_columns + present_optional_history]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_rows")
    .to_frame()
)

display(missing_summary)
print(
    "Optional N03 history columns present:",
    len(present_optional_history),
    "of",
    len(OPTIONAL_N03_HISTORY_COLUMNS),
)


,missing_rows
hist_peak_memory_p90,83158
hist_peak_memory_mean,83158
hist_peak_memory_p95,83158
hist_peak_memory_p99,83158
hist_peak_cpu_mean,80801
hist_peak_cpu_p90,80801
hist_peak_cpu_p95,80801
hist_peak_cpu_p99,80801
peak_memory_normalized,4495
requested_cpu_ncu,774


Optional N03 history columns present: 10 of 10


# Freeze Chronological 70/15/15 Boundaries

The existing shared split utility is used on execution rows to preserve the project's established split boundary semantics. Because complete `execution_start_time_us` groups are preserved, every occurrence belongs entirely to one split.


In [45]:
(
    train_exec_raw,
    val_exec_raw,
    test_exec_raw,
    split_metadata,
) = chronological_split(
    execution_features,
    time_col="execution_start_time_us",
    id_col="execution_id",
    train_fraction=0.70,
    val_fraction=0.15,
)

print("Train execution rows:", len(train_exec_raw))
print("Validation execution rows:", len(val_exec_raw))
print("Test execution rows:", len(test_exec_raw))


Train execution rows: 178858
Validation execution rows: 36703
Test execution rows: 38024


In [46]:
assert len(train_exec_raw) > 0
assert len(val_exec_raw) > 0
assert len(test_exec_raw) > 0

assert train_exec_raw["execution_id"].is_unique
assert val_exec_raw["execution_id"].is_unique
assert test_exec_raw["execution_id"].is_unique

assert set(train_exec_raw["execution_id"]).isdisjoint(
    set(val_exec_raw["execution_id"])
)
assert set(train_exec_raw["execution_id"]).isdisjoint(
    set(test_exec_raw["execution_id"])
)
assert set(val_exec_raw["execution_id"]).isdisjoint(
    set(test_exec_raw["execution_id"])
)

assert (
    train_exec_raw["execution_start_time_us"].max()
    < val_exec_raw["execution_start_time_us"].min()
)
assert (
    val_exec_raw["execution_start_time_us"].max()
    < test_exec_raw["execution_start_time_us"].min()
)
assert (
    len(train_exec_raw) + len(val_exec_raw) + len(test_exec_raw)
    == len(execution_features)
)

TRAIN_END_TIME = train_exec_raw["execution_start_time_us"].max()
VAL_START_TIME = val_exec_raw["execution_start_time_us"].min()
VAL_END_TIME = val_exec_raw["execution_start_time_us"].max()
TEST_START_TIME = test_exec_raw["execution_start_time_us"].min()

assert TRAIN_END_TIME < VAL_START_TIME
assert VAL_END_TIME < TEST_START_TIME

print("Split integrity and timestamp-boundary checks passed.")
print("Train end:", TRAIN_END_TIME)
print("Validation start:", VAL_START_TIME)
print("Validation end:", VAL_END_TIME)
print("Test start:", TEST_START_TIME)


Split integrity and timestamp-boundary checks passed.
Train end: 1812600000000
Validation start: 1813200000000
Validation end: 2261100000000
Test start: 2262600000000


In [47]:
total_execution_rows = len(execution_features)

execution_split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [
            len(train_exec_raw),
            len(val_exec_raw),
            len(test_exec_raw),
        ],
        "fraction": [
            len(train_exec_raw) / total_execution_rows,
            len(val_exec_raw) / total_execution_rows,
            len(test_exec_raw) / total_execution_rows,
        ],
        "start_time_us": [
            train_exec_raw["execution_start_time_us"].min(),
            val_exec_raw["execution_start_time_us"].min(),
            test_exec_raw["execution_start_time_us"].min(),
        ],
        "end_time_us": [
            TRAIN_END_TIME,
            VAL_END_TIME,
            test_exec_raw["execution_start_time_us"].max(),
        ],
    }
)

display(execution_split_summary)


,split,rows,fraction,start_time_us,end_time_us
0,train,178858,0.705318,300000000,1812600000000
1,validation,36703,0.144736,1813200000000,2261100000000
2,test,38024,0.149946,2262600000000,2678400000000


# Build Canonical Occurrences

An occurrence is `recurrence_key + execution_start_time_us`.

For simultaneous execution rows within an occurrence:
- occurrence CPU peak = maximum `peak_cpu_ncu`;
- occurrence memory peak = maximum `peak_memory_normalized`;
- occurrence request = maximum non-missing request for that resource;
- occurrence end = latest execution end;
- occurrence runtime = latest end minus occurrence start.

Using `max` for request and observed peak gives a deterministic conservative per-occurrence representation. Request heterogeneity is audited below.


In [48]:
def build_occurrence_table(df: pd.DataFrame) -> pd.DataFrame:
    grouped = (
        df.groupby(
            ["recurrence_key", "execution_start_time_us"],
            as_index=False,
            sort=True,
        )
        .agg(
            occurrence_end_time_us=("execution_end_time_us", "max"),
            occurrence_execution_count=("execution_id", "size"),
            occurrence_requested_cpu_ncu=("requested_cpu_ncu", "max"),
            occurrence_requested_memory_normalized=(
                "requested_memory_normalized", "max"
            ),
            occurrence_peak_cpu_ncu=("peak_cpu_ncu", "max"),
            occurrence_peak_memory_normalized=(
                "peak_memory_normalized", "max"
            ),
            occurrence_cpu_request_nunique=(
                "requested_cpu_ncu", "nunique"
            ),
            occurrence_memory_request_nunique=(
                "requested_memory_normalized", "nunique"
            ),
        )
        .sort_values(
            ["recurrence_key", "execution_start_time_us"],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    grouped["occurrence_runtime_us"] = (
        grouped["occurrence_end_time_us"]
        - grouped["execution_start_time_us"]
    )

    grouped["occurrence_id"] = (
        grouped["recurrence_key"].astype(str)
        + "||"
        + grouped["execution_start_time_us"].astype(str)
    )

    if grouped["occurrence_id"].duplicated().any():
        raise RuntimeError("occurrence_id is not unique.")

    return grouped


all_occurrences = build_occurrence_table(execution_features)

print("Execution rows:", len(execution_features))
print("Distinct occurrences:", len(all_occurrences))
print(
    "Multi-execution occurrences:",
    int((all_occurrences["occurrence_execution_count"] > 1).sum()),
)


Execution rows: 253585
Distinct occurrences: 4879
Multi-execution occurrences: 3307


In [49]:
occurrence_audit = pd.Series(
    {
        "distinct_occurrences": len(all_occurrences),
        "multi_execution_occurrences": int(
            (all_occurrences["occurrence_execution_count"] > 1).sum()
        ),
        "occurrences_with_heterogeneous_cpu_request": int(
            (all_occurrences["occurrence_cpu_request_nunique"] > 1).sum()
        ),
        "occurrences_with_heterogeneous_memory_request": int(
            (all_occurrences["occurrence_memory_request_nunique"] > 1).sum()
        ),
        "occurrences_missing_cpu_request": int(
            all_occurrences["occurrence_requested_cpu_ncu"].isna().sum()
        ),
        "occurrences_missing_memory_request": int(
            all_occurrences[
                "occurrence_requested_memory_normalized"
            ].isna().sum()
        ),
        "occurrences_missing_cpu_peak": int(
            all_occurrences["occurrence_peak_cpu_ncu"].isna().sum()
        ),
        "occurrences_missing_memory_peak": int(
            all_occurrences[
                "occurrence_peak_memory_normalized"
            ].isna().sum()
        ),
    },
    name="count",
)

display(occurrence_audit.to_frame())


,count
distinct_occurrences,4879
multi_execution_occurrences,3307
occurrences_with_heterogeneous_cpu_request,1918
occurrences_with_heterogeneous_memory_request,1997
occurrences_missing_cpu_request,3
occurrences_missing_memory_request,3
occurrences_missing_cpu_peak,0
occurrences_missing_memory_peak,60


# Build Past-Only Completed-Occurrence History

This is the key correction to the earlier N04.

For a current occurrence starting at time `t`, an earlier occurrence is eligible history only when its **occurrence end time is <= t**. Therefore:

- overlapping/not-yet-finished occurrences are excluded;
- `prior_completed_occurrence_count` counts occurrences, not execution rows;
- CPU/memory history counts are resource-specific counts of valid completed occurrence peaks;
- Mean/P90/P95/P99 are calculated from completed occurrence peaks only.

The older N03 execution-row history columns remain available in the source artifact but are not used for baseline eligibility or baseline values here.


In [50]:
def _numeric_history_stats(values: list[float]) -> dict[str, float]:
    if not values:
        return {
            "count": 0,
            "mean": np.nan,
            "std": np.nan,
            "p90": np.nan,
            "p95": np.nan,
            "p99": np.nan,
        }

    array = np.asarray(values, dtype=float)

    return {
        "count": int(array.size),
        "mean": float(array.mean()),
        "std": (
            float(array.std(ddof=1))
            if array.size > 1
            else np.nan
        ),
        "p90": float(np.quantile(array, 0.90)),
        "p95": float(np.quantile(array, 0.95)),
        "p99": float(np.quantile(array, 0.99)),
    }


def add_completed_occurrence_history(
    occurrence_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add history available at each occurrence start.

    Uses a completion-time heap so an occurrence enters history only after
    its latest execution row has completed.
    """

    ordered = (
        occurrence_df
        .sort_values(
            ["recurrence_key", "execution_start_time_us"],
            kind="mergesort",
        )
        .reset_index(drop=True)
        .copy()
    )

    history_rows = []

    for recurrence_key, group in ordered.groupby(
        "recurrence_key",
        sort=False,
    ):
        pending = []
        completed_occurrence_count = 0
        completed_cpu = []
        completed_memory = []
        completed_runtime = []
        sequence = 0

        for row in group.itertuples(index=False):
            current_start = row.execution_start_time_us

            # Move every earlier occurrence that has completed by the current
            # start time into the available-history pool.
            while pending and pending[0][0] <= current_start:
                (
                    _end_time,
                    _sequence,
                    cpu_peak,
                    memory_peak,
                    runtime_us,
                ) = heapq.heappop(pending)

                completed_occurrence_count += 1

                if pd.notna(cpu_peak):
                    completed_cpu.append(float(cpu_peak))
                if pd.notna(memory_peak):
                    completed_memory.append(float(memory_peak))
                if pd.notna(runtime_us):
                    completed_runtime.append(float(runtime_us))

            cpu_stats = _numeric_history_stats(completed_cpu)
            memory_stats = _numeric_history_stats(completed_memory)
            runtime_stats = _numeric_history_stats(completed_runtime)

            history_rows.append(
                {
                    "occurrence_id": row.occurrence_id,
                    "prior_completed_occurrence_count": (
                        completed_occurrence_count
                    ),
                    "occ_hist_peak_cpu_count": cpu_stats["count"],
                    "occ_hist_peak_cpu_mean": cpu_stats["mean"],
                    "occ_hist_peak_cpu_std": cpu_stats["std"],
                    "occ_hist_peak_cpu_p90": cpu_stats["p90"],
                    "occ_hist_peak_cpu_p95": cpu_stats["p95"],
                    "occ_hist_peak_cpu_p99": cpu_stats["p99"],
                    "occ_hist_peak_memory_count": memory_stats["count"],
                    "occ_hist_peak_memory_mean": memory_stats["mean"],
                    "occ_hist_peak_memory_std": memory_stats["std"],
                    "occ_hist_peak_memory_p90": memory_stats["p90"],
                    "occ_hist_peak_memory_p95": memory_stats["p95"],
                    "occ_hist_peak_memory_p99": memory_stats["p99"],
                    "occ_hist_runtime_count": runtime_stats["count"],
                    "occ_hist_runtime_mean": runtime_stats["mean"],
                    "occ_hist_runtime_std": runtime_stats["std"],
                    "occ_hist_runtime_p90": runtime_stats["p90"],
                    "occ_hist_runtime_p95": runtime_stats["p95"],
                    "occ_hist_runtime_p99": runtime_stats["p99"],
                }
            )

            # The current occurrence is not available to its own history.
            heapq.heappush(
                pending,
                (
                    row.occurrence_end_time_us,
                    sequence,
                    row.occurrence_peak_cpu_ncu,
                    row.occurrence_peak_memory_normalized,
                    row.occurrence_runtime_us,
                ),
            )
            sequence += 1

    history_df = pd.DataFrame(history_rows)

    result = ordered.merge(
        history_df,
        on="occurrence_id",
        how="left",
        validate="one_to_one",
    )

    return result


all_occurrences = add_completed_occurrence_history(all_occurrences)
print("Completed-occurrence history added.")


Completed-occurrence history added.


In [51]:
assert (
    all_occurrences["occ_hist_peak_cpu_count"]
    <= all_occurrences["prior_completed_occurrence_count"]
).all()
assert (
    all_occurrences["occ_hist_peak_memory_count"]
    <= all_occurrences["prior_completed_occurrence_count"]
).all()
assert (
    all_occurrences["occ_hist_runtime_count"]
    <= all_occurrences["prior_completed_occurrence_count"]
).all()

history_audit = pd.Series(
    {
        "max_prior_completed_occurrences": int(
            all_occurrences["prior_completed_occurrence_count"].max()
        ),
        "rows_where_cpu_valid_count_differs_from_completed_count": int(
            (
                all_occurrences["occ_hist_peak_cpu_count"]
                != all_occurrences["prior_completed_occurrence_count"]
            ).sum()
        ),
        "rows_where_memory_valid_count_differs_from_completed_count": int(
            (
                all_occurrences["occ_hist_peak_memory_count"]
                != all_occurrences["prior_completed_occurrence_count"]
            ).sum()
        ),
    },
    name="count",
)

display(history_audit.to_frame())

display(
    all_occurrences[
        [
            "occurrence_id",
            "recurrence_key",
            "execution_start_time_us",
            "occurrence_end_time_us",
            "prior_completed_occurrence_count",
            "occ_hist_peak_cpu_count",
            "occ_hist_peak_memory_count",
        ]
    ].head(20)
)


,count
max_prior_completed_occurrences,273
rows_where_cpu_valid_count_differs_from_completed_count,0
rows_where_memory_valid_count_differs_from_completed_count,40


,occurrence_id,recurrence_key,execution_start_time_us,occurrence_end_time_us,prior_completed_occurrence_count,occ_hist_peak_cpu_count,occ_hist_peak_memory_count
0,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,2097300000000,2097600000000,0,0,0
1,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,116400000000,116700000000,0,0,0
2,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,++iQVWDf5/Hvnn/NXU3ttQKP/dLrqRkP3MLiZn1jKdA=||...,359400000000,359700000000,0,0,0
3,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,45600000000,45900000000,0,0,0
4,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,+1sqV4mrzK9pDfrhUsBxHQu9o/DuKJAFdsXhg31E9rU=||...,1200900000000,1201200000000,1,1,1
5,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,2303700000000,2304000000000,0,0,0
6,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,+52peVL4nAGr4mgVjZQpkJDhArvSYmdIoludPEovCFU=||...,1165200000000,1165500000000,0,0,0
7,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,1272165000000,1272168000000,0,0,0
8,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,892800000000,893100000000,0,0,0
9,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,+9EArMpTcRKY6SZ+doAw1QllovlleQnzrJD843XiIDo=||...,1726200000000,1726500000000,0,0,0


# Apply Frozen Split Boundaries to Occurrences

History was constructed globally using only backwards-looking completion-time information. This allows validation/test occurrences to use legitimate earlier history from preceding splits without allowing future information into the past.


In [52]:
def split_occurrences_by_frozen_boundaries(
    occurrence_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_mask = (
        occurrence_df["execution_start_time_us"] <= TRAIN_END_TIME
    )
    val_mask = (
        (occurrence_df["execution_start_time_us"] >= VAL_START_TIME)
        & (occurrence_df["execution_start_time_us"] <= VAL_END_TIME)
    )
    test_mask = (
        occurrence_df["execution_start_time_us"] >= TEST_START_TIME
    )

    assigned = train_mask.astype(int) + val_mask.astype(int) + test_mask.astype(int)
    if not (assigned == 1).all():
        raise RuntimeError(
            "Every occurrence must map to exactly one frozen split."
        )

    return (
        occurrence_df.loc[train_mask].reset_index(drop=True),
        occurrence_df.loc[val_mask].reset_index(drop=True),
        occurrence_df.loc[test_mask].reset_index(drop=True),
    )


train_occurrences, val_occurrences, test_occurrences = (
    split_occurrences_by_frozen_boundaries(all_occurrences)
)

print("Train occurrences:", len(train_occurrences))
print("Validation occurrences:", len(val_occurrences))
print("Test occurrences:", len(test_occurrences))


Train occurrences: 3473
Validation occurrences: 675
Test occurrences: 731


In [53]:
def execution_occurrence_count(df: pd.DataFrame) -> int:
    return len(
        df[["recurrence_key", "execution_start_time_us"]]
        .drop_duplicates()
    )

assert len(train_occurrences) == execution_occurrence_count(train_exec_raw)
assert len(val_occurrences) == execution_occurrence_count(val_exec_raw)
assert len(test_occurrences) == execution_occurrence_count(test_exec_raw)

occurrence_split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "occurrences": [
            len(train_occurrences),
            len(val_occurrences),
            len(test_occurrences),
        ],
        "multi_execution_occurrences": [
            int((train_occurrences["occurrence_execution_count"] > 1).sum()),
            int((val_occurrences["occurrence_execution_count"] > 1).sum()),
            int((test_occurrences["occurrence_execution_count"] > 1).sum()),
        ],
    }
)

display(occurrence_split_summary)


,split,occurrences,multi_execution_occurrences
0,train,3473,2277
1,validation,675,513
2,test,731,517


# Next-Occurrence Target Construction

Targets are created at the occurrence level and independently inside train, validation, and test.

This prevents:
- naïve row-wise shifting across simultaneous execution rows; and
- a training target from pointing into validation or a validation target from pointing into test.


In [54]:
TARGET_CPU_COL = "target_cpu_next_occurrence_peak_ncu"
TARGET_MEMORY_COL = "target_memory_next_occurrence_peak_normalized"
TARGET_TIME_COL = "target_next_occurrence_start_time_us"
TARGET_OCCURRENCE_SIZE_COL = "target_next_occurrence_execution_count"


def add_next_occurrence_targets(
    occurrence_df: pd.DataFrame,
) -> pd.DataFrame:
    result = (
        occurrence_df
        .sort_values(
            ["recurrence_key", "execution_start_time_us"],
            kind="mergesort",
        )
        .reset_index(drop=True)
        .copy()
    )

    groups = result.groupby("recurrence_key", sort=False)

    result[TARGET_TIME_COL] = (
        groups["execution_start_time_us"].shift(-1)
    )
    result[TARGET_CPU_COL] = (
        groups["occurrence_peak_cpu_ncu"].shift(-1)
    )
    result[TARGET_MEMORY_COL] = (
        groups["occurrence_peak_memory_normalized"].shift(-1)
    )
    result[TARGET_OCCURRENCE_SIZE_COL] = (
        groups["occurrence_execution_count"].shift(-1)
    )

    target_available = result[TARGET_TIME_COL].notna()
    if target_available.any():
        assert (
            result.loc[target_available, TARGET_TIME_COL]
            > result.loc[target_available, "execution_start_time_us"]
        ).all()

    return result


train_occurrences = add_next_occurrence_targets(train_occurrences)
val_occurrences = add_next_occurrence_targets(val_occurrences)

print(
    "Train and validation next-occurrence targets created "
    "at occurrence level."
)


Train and validation next-occurrence targets created at occurrence level.


In [55]:
def build_target_audit(split_name: str, df: pd.DataFrame) -> dict[str, object]:
    return {
        "split": split_name,
        "occurrences": len(df),
        "occurrences_with_next_occurrence": int(df[TARGET_TIME_COL].notna().sum()),
        "occurrences_with_cpu_target": int(df[TARGET_CPU_COL].notna().sum()),
        "occurrences_with_memory_target": int(df[TARGET_MEMORY_COL].notna().sum()),
        "multi_execution_occurrences": int(
            (df["occurrence_execution_count"] > 1).sum()
        ),
    }


target_audit = pd.DataFrame(
    [
        build_target_audit("train", train_occurrences),
        build_target_audit("validation", val_occurrences),
    ]
)

display(target_audit)


,split,occurrences,occurrences_with_next_occurrence,occurrences_with_cpu_target,occurrences_with_memory_target,multi_execution_occurrences
0,train,3473,1672,1672,1649,2277
1,validation,675,226,226,225,513


In [56]:
# Targets must stay within their frozen split.
assert train_occurrences[TARGET_TIME_COL].dropna().le(TRAIN_END_TIME).all()
assert val_occurrences[TARGET_TIME_COL].dropna().between(
    VAL_START_TIME,
    VAL_END_TIME,
).all()

print("Target split-isolation checks passed.")


Target split-isolation checks passed.


# Occurrence-Level Baseline Recommendations

Policies: Request, Mean, P90, P95, P99.

For historical policies, a resource uses history only when at least `min_runs` **valid completed occurrences for that resource** are available. Otherwise it falls back to the current occurrence request.

This means `min_runs` is no longer based on N03 execution-row counts.


In [57]:
CPU_BASELINE_COLUMNS = {
    "mean": "occ_hist_peak_cpu_mean",
    "p90": "occ_hist_peak_cpu_p90",
    "p95": "occ_hist_peak_cpu_p95",
    "p99": "occ_hist_peak_cpu_p99",
}

MEMORY_BASELINE_COLUMNS = {
    "mean": "occ_hist_peak_memory_mean",
    "p90": "occ_hist_peak_memory_p90",
    "p95": "occ_hist_peak_memory_p95",
    "p99": "occ_hist_peak_memory_p99",
}

CPU_REQUEST_COL = "occurrence_requested_cpu_ncu"
MEMORY_REQUEST_COL = "occurrence_requested_memory_normalized"
CPU_HISTORY_COUNT_COL = "occ_hist_peak_cpu_count"
MEMORY_HISTORY_COUNT_COL = "occ_hist_peak_memory_count"


def add_baseline_recommendations(
    df: pd.DataFrame,
    min_runs: int,
) -> pd.DataFrame:
    if min_runs < 1:
        raise ValueError("min_runs must be at least 1.")

    result = df.copy()

    # Request baseline = carry forward the current occurrence request.
    result["rec_cpu_request"] = result[CPU_REQUEST_COL]
    result["rec_memory_request"] = result[MEMORY_REQUEST_COL]

    for policy_name, history_col in CPU_BASELINE_COLUMNS.items():
        enough_history = result[CPU_HISTORY_COUNT_COL] >= min_runs
        historical_value_available = result[history_col].notna()
        use_history = enough_history & historical_value_available

        result[f"uses_history_cpu_{policy_name}"] = use_history
        result[f"rec_cpu_{policy_name}"] = result[history_col].where(
            use_history,
            result[CPU_REQUEST_COL],
        )

    for policy_name, history_col in MEMORY_BASELINE_COLUMNS.items():
        enough_history = result[MEMORY_HISTORY_COUNT_COL] >= min_runs
        historical_value_available = result[history_col].notna()
        use_history = enough_history & historical_value_available

        result[f"uses_history_memory_{policy_name}"] = use_history
        result[f"rec_memory_{policy_name}"] = result[history_col].where(
            use_history,
            result[MEMORY_REQUEST_COL],
        )

    return result


## Build Validation Candidates


In [58]:
validation_candidates = {}

for min_runs in MIN_RUNS_CANDIDATES:
    validation_candidates[min_runs] = add_baseline_recommendations(
        val_occurrences,
        min_runs=min_runs,
    )
    print(
        f"min_runs={min_runs}:",
        validation_candidates[min_runs].shape,
    )

assert all(
    len(candidate_df) == len(val_occurrences)
    for candidate_df in validation_candidates.values()
)


min_runs=1: (675, 53)
min_runs=3: (675, 53)
min_runs=5: (675, 53)
min_runs=10: (675, 53)


In [59]:
fallback_summary_rows = []

for min_runs, candidate_df in validation_candidates.items():
    cpu_fallback = candidate_df[CPU_HISTORY_COUNT_COL] < min_runs
    memory_fallback = candidate_df[MEMORY_HISTORY_COUNT_COL] < min_runs

    fallback_summary_rows.append(
        {
            "min_runs": min_runs,
            "validation_occurrences": len(candidate_df),
            "cpu_fallback_occurrences": int(cpu_fallback.sum()),
            "cpu_fallback_pct": 100 * float(cpu_fallback.mean()),
            "memory_fallback_occurrences": int(memory_fallback.sum()),
            "memory_fallback_pct": 100 * float(memory_fallback.mean()),
        }
    )

fallback_summary = pd.DataFrame(fallback_summary_rows)
display(fallback_summary)


,min_runs,validation_occurrences,cpu_fallback_occurrences,cpu_fallback_pct,memory_fallback_occurrences,memory_fallback_pct
0,1,675,239,35.407407,242,35.851852
1,3,675,378,56.000000,384,56.888889
2,5,675,436,64.592593,440,65.185185
3,10,675,506,74.962963,510,75.555556


# Shared Evaluator Integration

**Critical alignment:** every policy is evaluated against the same next-occurrence CPU/memory targets, not the current occurrence peaks.

A single common validation population is frozen before policy scoring. It requires both next-occurrence targets and both Request recommendations to be available. Because every historical policy falls back to Request, this gives Request/Mean/P90/P95/P99 the same occurrence denominator.


In [60]:
from src.evaluation.evaluator import RecommendationEvaluator


evaluator = RecommendationEvaluator()

BASELINE_POLICIES = {
    "Request": ("rec_cpu_request", "rec_memory_request"),
    "Mean": ("rec_cpu_mean", "rec_memory_mean"),
    "P90": ("rec_cpu_p90", "rec_memory_p90"),
    "P95": ("rec_cpu_p95", "rec_memory_p95"),
    "P99": ("rec_cpu_p99", "rec_memory_p99"),
}

# All baselines are scored on the SAME future targets.
ACTUAL_CPU_COL = TARGET_CPU_COL
ACTUAL_MEMORY_COL = TARGET_MEMORY_COL

BASELINE_POLICIES


{'Request': ('rec_cpu_request', 'rec_memory_request'),
 'Mean': ('rec_cpu_mean', 'rec_memory_mean'),
 'P90': ('rec_cpu_p90', 'rec_memory_p90'),
 'P95': ('rec_cpu_p95', 'rec_memory_p95'),
 'P99': ('rec_cpu_p99', 'rec_memory_p99')}

In [61]:
# Common validation occurrence population shared by every policy/min_runs.
base_validation = validation_candidates[MIN_RUNS_CANDIDATES[0]]

COMMON_EVAL_REQUIRED = [
    TARGET_CPU_COL,
    TARGET_MEMORY_COL,
    "rec_cpu_request",
    "rec_memory_request",
]

validation_common_mask = (
    base_validation[COMMON_EVAL_REQUIRED]
    .notna()
    .all(axis=1)
)

validation_eval_occurrence_ids = set(
    base_validation.loc[
        validation_common_mask,
        "occurrence_id",
    ]
)

print("Validation occurrences total:", len(base_validation))
print(
    "Common evaluated occurrences:",
    len(validation_eval_occurrence_ids),
)
print(
    "Excluded from common evaluation:",
    len(base_validation) - len(validation_eval_occurrence_ids),
)

if not validation_eval_occurrence_ids:
    raise RuntimeError("No validation occurrences are eligible for evaluation.")


Validation occurrences total: 675
Common evaluated occurrences: 225
Excluded from common evaluation: 450


In [62]:
# Confirm that the updated evaluator exposes explicit valid denominators.

sanity_candidate = validation_candidates[MIN_RUNS_CANDIDATES[0]]

sanitary = sanity_candidate[
    sanity_candidate["occurrence_id"].isin(
        validation_eval_occurrence_ids
    )
].copy()

sanity_evaluated = evaluator.evaluate(
    data=sanitary,
    actual_cpu_col=ACTUAL_CPU_COL,
    actual_memory_col=ACTUAL_MEMORY_COL,
    recommended_cpu_col="rec_cpu_request",
    recommended_memory_col="rec_memory_request",
    policy_name="Request",
    id_cols=["occurrence_id", "recurrence_key"],
)

sanity_summary = evaluator.summarize(sanity_evaluated)


# ------------------------------------------------------------
# Valid-count definitions
# ------------------------------------------------------------

# These counts MUST be identical across every policy/min_runs
# configuration because violation evaluation uses the same
# occurrence population and the same next-occurrence targets.
SHARED_VALID_COUNT_KEYS = {
    "valid_cpu_violation_count",
    "valid_memory_violation_count",
    "valid_any_resource_violation_count",
}


# These counts may legitimately differ between policies.
#
# Utilization typically uses the recommendation as a denominator,
# so a zero/non-positive recommendation can make utilization
# undefined for one policy while still being valid for another.
POLICY_DEPENDENT_VALID_COUNT_KEYS = {
    "valid_cpu_utilization_count",
    "valid_memory_utilization_count",
}


# We still require the evaluator to expose all of these fields.
EXPECTED_VALID_COUNT_KEYS = (
    SHARED_VALID_COUNT_KEYS
    | POLICY_DEPENDENT_VALID_COUNT_KEYS
)


# ------------------------------------------------------------
# Verify evaluator version
# ------------------------------------------------------------

missing_evaluator_fields = (
    EXPECTED_VALID_COUNT_KEYS
    - set(sanity_summary.keys())
)

if missing_evaluator_fields:
    raise RuntimeError(
        "The updated shared evaluator is not loaded. "
        "Merge the latest main branch, restart the kernel, "
        "and rerun. Missing summary fields: "
        f"{sorted(missing_evaluator_fields)}"
    )


# ------------------------------------------------------------
# Basic sanity checks
# ------------------------------------------------------------

if len(sanitary) != len(validation_eval_occurrence_ids):
    raise RuntimeError(
        "Sanity-check dataframe does not contain exactly the "
        "common validation occurrence population."
    )


for count_key in EXPECTED_VALID_COUNT_KEYS:
    count_value = sanity_summary[count_key]

    if pd.isna(count_value):
        raise RuntimeError(
            f"Evaluator returned NaN for {count_key}."
        )

    if count_value < 0:
        raise RuntimeError(
            f"Evaluator returned a negative count for {count_key}: "
            f"{count_value}"
        )

    if count_value > len(sanitary):
        raise RuntimeError(
            f"Evaluator count {count_key}={count_value} exceeds "
            f"the common validation population of {len(sanitary)}."
        )


# ------------------------------------------------------------
# Display evaluator sanity information
# ------------------------------------------------------------

print("Updated evaluator detected.")
print(
    "Common input occurrences:",
    len(sanitary),
)

print(
    "Shared violation denominators:",
    {
        key: sanity_summary[key]
        for key in sorted(SHARED_VALID_COUNT_KEYS)
    },
)

print(
    "Policy-dependent utilization denominators:",
    {
        key: sanity_summary[key]
        for key in sorted(POLICY_DEPENDENT_VALID_COUNT_KEYS)
    },
)

display(
    pd.Series(
        sanity_summary,
        name="Request",
    ).to_frame()
)

Updated evaluator detected.
Common input occurrences: 225
Shared violation denominators: {'valid_any_resource_violation_count': 225, 'valid_cpu_violation_count': 225, 'valid_memory_violation_count': 225}
Policy-dependent utilization denominators: {'valid_cpu_utilization_count': 221, 'valid_memory_utilization_count': 225}


,Request
row_count,225.000000
valid_cpu_violation_count,225.000000
valid_memory_violation_count,225.000000
valid_any_resource_violation_count,225.000000
valid_cpu_utilization_count,221.000000
valid_memory_utilization_count,225.000000
cpu_violation_pct,56.888889
memory_violation_pct,29.777778
any_resource_violation_pct,61.333333
average_cpu_utilization,0.673638


## Evaluate All Validation Configurations


In [63]:
validation_evaluation_rows = []
evaluated_validation = {}

for min_runs in MIN_RUNS_CANDIDATES:
    candidate_df = validation_candidates[min_runs]
    eval_df = candidate_df[
        candidate_df["occurrence_id"].isin(validation_eval_occurrence_ids)
    ].copy()

    if len(eval_df) != len(validation_eval_occurrence_ids):
        raise RuntimeError(
            "Validation candidate does not preserve the common occurrence population."
        )

    for policy_name, (
        cpu_recommendation_col,
        memory_recommendation_col,
    ) in BASELINE_POLICIES.items():
        recommendation_cols = [
            cpu_recommendation_col,
            memory_recommendation_col,
        ]
        if eval_df[recommendation_cols].isna().any().any():
            raise RuntimeError(
                f"{policy_name} contains missing recommendations on the "
                "common evaluation population."
            )

        evaluated = evaluator.evaluate(
            data=eval_df,
            actual_cpu_col=ACTUAL_CPU_COL,
            actual_memory_col=ACTUAL_MEMORY_COL,
            recommended_cpu_col=cpu_recommendation_col,
            recommended_memory_col=memory_recommendation_col,
            policy_name=policy_name,
            id_cols=["occurrence_id", "recurrence_key"],
        )

        summary = evaluator.summarize(evaluated)

        validation_evaluation_rows.append(
            {
                "min_runs": min_runs,
                "policy": policy_name,
                "n_common_evaluation_occurrences": len(eval_df),
                **summary,
            }
        )

        evaluated_validation[(min_runs, policy_name)] = evaluated

validation_comparison = (
    pd.DataFrame(validation_evaluation_rows)
    .sort_values(["min_runs", "policy"])
    .reset_index(drop=True)
)

display(validation_comparison)


,min_runs,policy,n_common_evaluation_occurrences,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,1,Mean,225,225,225,225,225,225,225,39.555556,39.111111,52.444444,0.613073,0.684142,0.032621,0.011557,0.049763,0.006268
1,1,P90,225,225,225,225,225,225,225,24.444444,20.444444,31.111111,0.469157,0.547696,0.092034,0.020081,0.035907,0.004708
2,1,P95,225,225,225,225,225,225,225,19.111111,16.000000,24.888889,0.405909,0.489733,0.165285,0.032564,0.024535,0.003338
3,1,P99,225,225,225,225,225,225,225,14.222222,13.333333,21.333333,0.351501,0.429056,0.352146,0.245287,0.013613,0.001827
4,1,Request,225,225,225,225,225,221,225,56.888889,29.777778,61.333333,0.673638,0.569585,0.019422,0.011725,0.068593,0.009155
5,3,Mean,225,225,225,225,225,224,225,38.666667,35.111111,49.777778,0.594725,0.650332,0.033799,0.013924,0.050709,0.005991
6,3,P90,225,225,225,225,225,224,225,23.555556,16.888889,28.888889,0.452170,0.516633,0.092983,0.022438,0.036899,0.004453
7,3,P95,225,225,225,225,225,224,225,18.222222,12.444444,22.666667,0.388799,0.458974,0.166206,0.034919,0.025533,0.003085
8,3,P99,225,225,225,225,225,224,225,13.333333,9.777778,19.111111,0.334265,0.398524,0.353044,0.247641,0.014615,0.001577
9,3,Request,225,225,225,225,225,221,225,56.888889,29.777778,61.333333,0.673638,0.569585,0.019422,0.011725,0.068593,0.009155


In [64]:
assert len(validation_comparison) == (
    len(MIN_RUNS_CANDIDATES) * len(BASELINE_POLICIES)
)

# Every configuration receives exactly the same source occurrence population.
assert validation_comparison[
    "n_common_evaluation_occurrences"
].nunique() == 1


# Violation denominators MUST be identical across every policy/min_runs
# configuration because they use the same occurrences and the same targets.
for count_key in SHARED_VALID_COUNT_KEYS:
    if validation_comparison[count_key].nunique(dropna=False) != 1:
        raise RuntimeError(
            "Evaluator valid violation denominator differs across policies "
            f"for {count_key}."
        )


# Utilization denominators may differ across policies because utilization
# can be undefined when a recommendation is zero/non-positive.
# They still must be valid counts within the common source population.
for count_key in POLICY_DEPENDENT_VALID_COUNT_KEYS:
    if validation_comparison[count_key].isna().any():
        raise RuntimeError(
            f"{count_key} contains missing values."
        )

    if (
        validation_comparison[count_key]
        > validation_comparison["n_common_evaluation_occurrences"]
    ).any():
        raise RuntimeError(
            f"{count_key} exceeds the common evaluation population."
        )

    if (validation_comparison[count_key] < 0).any():
        raise RuntimeError(
            f"{count_key} contains a negative count."
        )


# Request must be invariant across min_runs candidates.
request_validation_results = (
    validation_comparison[
        validation_comparison["policy"] == "Request"
    ]
    .sort_values("min_runs")
    .reset_index(drop=True)
)

request_metric_columns = [
    column
    for column in request_validation_results.columns
    if column != "min_runs"
]

request_variation = request_validation_results[
    request_metric_columns
].nunique(dropna=False)

assert (request_variation <= 1).all()

print(
    "Validation comparison uses one shared source population."
)
print(
    "Violation denominators are identical across all configurations."
)
print(
    "Utilization denominators were validated separately because they "
    "may legitimately depend on the recommendation."
)
print(
    "Request baseline is invariant across min_runs candidates."
)

display(request_validation_results)

Validation comparison uses one shared source population.
Violation denominators are identical across all configurations.
Utilization denominators were validated separately because they may legitimately depend on the recommendation.
Request baseline is invariant across min_runs candidates.


,min_runs,policy,n_common_evaluation_occurrences,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,1,Request,225,225,225,225,225,221,225,56.888889,29.777778,61.333333,0.673638,0.569585,0.019422,0.011725,0.068593,0.009155
1,3,Request,225,225,225,225,225,221,225,56.888889,29.777778,61.333333,0.673638,0.569585,0.019422,0.011725,0.068593,0.009155
2,5,Request,225,225,225,225,225,221,225,56.888889,29.777778,61.333333,0.673638,0.569585,0.019422,0.011725,0.068593,0.009155
3,10,Request,225,225,225,225,225,221,225,56.888889,29.777778,61.333333,0.673638,0.569585,0.019422,0.011725,0.068593,0.009155


In [65]:
# Convenient focused view of the conservative historical baseline.
p99_validation_results = (
    validation_comparison[
        validation_comparison["policy"] == "P99"
    ]
    .sort_values("min_runs")
    .reset_index(drop=True)
)

display(p99_validation_results)


,min_runs,policy,n_common_evaluation_occurrences,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,1,P99,225,225,225,225,225,225,225,14.222222,13.333333,21.333333,0.351501,0.429056,0.352146,0.245287,0.013613,0.001827
1,3,P99,225,225,225,225,225,224,225,13.333333,9.777778,19.111111,0.334265,0.398524,0.353044,0.247641,0.014615,0.001577
2,5,P99,225,225,225,225,225,223,225,16.444444,8.888889,21.333333,0.351911,0.387519,0.351349,0.248314,0.017044,0.001418
3,10,P99,225,225,225,225,225,221,225,23.111111,8.888889,27.111111,0.390650,0.376117,0.339439,0.247001,0.019177,0.001917


# Validation-Only Baseline Selection

The previous notebook hard-coded `P99 + min_runs=5` based on an evaluation that mixed execution-row history and current-occurrence outcomes. That result is **not reused** here.

The corrected notebook selects from Mean/P90/P95/P99 using validation only. The ranking is violation-focused:

1. lowest any-resource violation percentage;
2. lowest worst single-resource violation percentage;
3. lowest sum of CPU + memory violation percentages;
4. lower CPU waste;
5. lower memory waste;
6. lower `min_runs` as the final tie-breaker.

Request remains a comparator and is not selected as the historical baseline policy.


In [66]:
selection_candidates = validation_comparison[
    validation_comparison["policy"] != "Request"
].copy()

selection_candidates["max_resource_violation_pct"] = (
    selection_candidates[
        ["cpu_violation_pct", "memory_violation_pct"]
    ].max(axis=1)
)
selection_candidates["sum_resource_violation_pct"] = (
    selection_candidates["cpu_violation_pct"]
    + selection_candidates["memory_violation_pct"]
)

selection_sort_columns = [
    "any_resource_violation_pct",
    "max_resource_violation_pct",
    "sum_resource_violation_pct",
    "average_cpu_waste",
    "average_memory_waste",
    "min_runs",
    "policy",
]

selection_candidates = selection_candidates.sort_values(
    selection_sort_columns,
    kind="mergesort",
).reset_index(drop=True)

selected_validation_result = selection_candidates.iloc[[0]].copy()
SELECTED_MIN_RUNS = int(
    selected_validation_result.iloc[0]["min_runs"]
)
SELECTED_BASELINE_POLICY = str(
    selected_validation_result.iloc[0]["policy"]
)

print("Selected min_runs:", SELECTED_MIN_RUNS)
print("Selected historical baseline policy:", SELECTED_BASELINE_POLICY)
display(selection_candidates.head(10))
display(selected_validation_result)


Selected min_runs: 3
Selected historical baseline policy: P99


,min_runs,policy,n_common_evaluation_occurrences,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall,max_resource_violation_pct,sum_resource_violation_pct
0,3,P99,225,225,225,225,225,224,225,13.333333,9.777778,19.111111,0.334265,0.398524,0.353044,0.247641,0.014615,0.001577,13.333333,23.111111
1,1,P99,225,225,225,225,225,225,225,14.222222,13.333333,21.333333,0.351501,0.429056,0.352146,0.245287,0.013613,0.001827,14.222222,27.555556
2,5,P99,225,225,225,225,225,223,225,16.444444,8.888889,21.333333,0.351911,0.387519,0.351349,0.248314,0.017044,0.001418,16.444444,25.333333
3,3,P95,225,225,225,225,225,224,225,18.222222,12.444444,22.666667,0.388799,0.458974,0.166206,0.034919,0.025533,0.003085,18.222222,30.666667
4,1,P95,225,225,225,225,225,225,225,19.111111,16.000000,24.888889,0.405909,0.489733,0.165285,0.032564,0.024535,0.003338,19.111111,35.111111
5,5,P95,225,225,225,225,225,223,225,21.333333,11.555556,24.888889,0.405720,0.446965,0.164889,0.035643,0.027947,0.002923,21.333333,32.888889
6,10,P99,225,225,225,225,225,221,225,23.111111,8.888889,27.111111,0.390650,0.376117,0.339439,0.247001,0.019177,0.001917,23.111111,32.000000
7,3,P90,225,225,225,225,225,224,225,23.555556,16.888889,28.888889,0.452170,0.516633,0.092983,0.022438,0.036899,0.004453,23.555556,40.444444
8,10,P95,225,225,225,225,225,221,225,26.666667,11.555556,29.777778,0.440990,0.431430,0.155641,0.035541,0.029802,0.003378,26.666667,38.222222
9,5,P90,225,225,225,225,225,223,225,26.222222,15.111111,30.222222,0.468200,0.503324,0.092083,0.023224,0.039241,0.004289,26.222222,41.333333


,min_runs,policy,n_common_evaluation_occurrences,row_count,valid_cpu_violation_count,valid_memory_violation_count,valid_any_resource_violation_count,valid_cpu_utilization_count,valid_memory_utilization_count,cpu_violation_pct,memory_violation_pct,any_resource_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall,max_resource_violation_pct,sum_resource_violation_pct
0,3,P99,225,225,225,225,225,224,225,13.333333,9.777778,19.111111,0.334265,0.398524,0.353044,0.247641,0.014615,0.001577,13.333333,23.111111


In [67]:
# Validation choice is now locked.
# Materialize test targets only after model/baseline selection.

test_occurrences = add_next_occurrence_targets(test_occurrences)

assert (
    test_occurrences[TARGET_TIME_COL]
    .dropna()
    .ge(TEST_START_TIME)
    .all()
)

test_target_audit = pd.DataFrame(
    [
        build_target_audit(
            "test",
            test_occurrences,
        )
    ]
)

print(
    "Test targets materialized after validation configuration was locked."
)
print(
    "No test policy performance has been evaluated."
)

display(test_target_audit)

Test targets materialized after validation configuration was locked.
No test policy performance has been evaluated.


,split,occurrences,occurrences_with_next_occurrence,occurrences_with_cpu_target,occurrences_with_memory_target,multi_execution_occurrences
0,test,731,259,259,259,517


# Generate Final Baselines for All Occurrence Splits

Only after the validation choice is locked do we materialize selected-threshold recommendations for train, validation, and test. The test split is **not scored** in this notebook.


In [68]:
train_occurrences_final = add_baseline_recommendations(
    train_occurrences,
    min_runs=SELECTED_MIN_RUNS,
)
val_occurrences_final = add_baseline_recommendations(
    val_occurrences,
    min_runs=SELECTED_MIN_RUNS,
)
test_occurrences_final = add_baseline_recommendations(
    test_occurrences,
    min_runs=SELECTED_MIN_RUNS,
)

assert len(train_occurrences_final) == len(train_occurrences)
assert len(val_occurrences_final) == len(val_occurrences)
assert len(test_occurrences_final) == len(test_occurrences)

print("Final baseline columns generated for all occurrence splits.")
print("Test performance has not been evaluated.")


Final baseline columns generated for all occurrence splits.
Test performance has not been evaluated.


# Merge Canonical Occurrence Annotations Back to Execution Rows

These execution-row artifacts preserve compatibility and traceability. They are **not** the canonical evaluation population. N05/N06 should use the occurrence-level artifacts for model rows and policy metrics unless the modeling contract is deliberately changed and versioned.


In [69]:
OCCURRENCE_ANNOTATION_COLUMNS = [
    "occurrence_id",
    "occurrence_end_time_us",
    "occurrence_execution_count",
    "occurrence_requested_cpu_ncu",
    "occurrence_requested_memory_normalized",
    "occurrence_peak_cpu_ncu",
    "occurrence_peak_memory_normalized",
    "occurrence_runtime_us",
    "prior_completed_occurrence_count",
    "occ_hist_peak_cpu_count",
    "occ_hist_peak_cpu_mean",
    "occ_hist_peak_cpu_std",
    "occ_hist_peak_cpu_p90",
    "occ_hist_peak_cpu_p95",
    "occ_hist_peak_cpu_p99",
    "occ_hist_peak_memory_count",
    "occ_hist_peak_memory_mean",
    "occ_hist_peak_memory_std",
    "occ_hist_peak_memory_p90",
    "occ_hist_peak_memory_p95",
    "occ_hist_peak_memory_p99",
    "occ_hist_runtime_count",
    "occ_hist_runtime_mean",
    "occ_hist_runtime_std",
    "occ_hist_runtime_p90",
    "occ_hist_runtime_p95",
    "occ_hist_runtime_p99",
    TARGET_TIME_COL,
    TARGET_CPU_COL,
    TARGET_MEMORY_COL,
    TARGET_OCCURRENCE_SIZE_COL,
    "rec_cpu_request",
    "rec_cpu_mean",
    "rec_cpu_p90",
    "rec_cpu_p95",
    "rec_cpu_p99",
    "rec_memory_request",
    "rec_memory_mean",
    "rec_memory_p90",
    "rec_memory_p95",
    "rec_memory_p99",
]


def merge_occurrence_annotations(
    execution_df: pd.DataFrame,
    occurrence_df: pd.DataFrame,
) -> pd.DataFrame:
    working = execution_df.copy()
    working["_original_row_order"] = np.arange(len(working))

    mapping = occurrence_df[
        ["recurrence_key", "execution_start_time_us"]
        + OCCURRENCE_ANNOTATION_COLUMNS
    ].copy()

    result = working.merge(
        mapping,
        on=["recurrence_key", "execution_start_time_us"],
        how="left",
        validate="many_to_one",
    )

    result = (
        result
        .sort_values("_original_row_order", kind="mergesort")
        .drop(columns=["_original_row_order"])
        .reset_index(drop=True)
    )

    if result["occurrence_id"].isna().any():
        raise RuntimeError(
            "Some execution rows failed to map to an occurrence."
        )

    return result


train_features_final = merge_occurrence_annotations(
    train_exec_raw,
    train_occurrences_final,
)
val_features_final = merge_occurrence_annotations(
    val_exec_raw,
    val_occurrences_final,
)
test_features_final = merge_occurrence_annotations(
    test_exec_raw,
    test_occurrences_final,
)

assert len(train_features_final) == len(train_exec_raw)
assert len(val_features_final) == len(val_exec_raw)
assert len(test_features_final) == len(test_exec_raw)

print("Occurrence annotations merged back to execution-row artifacts.")


Occurrence annotations merged back to execution-row artifacts.


# Metadata and Canonical Artifact Save


In [70]:
def make_json_serializable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {
            key: make_json_serializable(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [make_json_serializable(item) for item in value]
    return value


In [71]:
metadata = {
    "dataset": "google_subset_v1",
    "source_artifact": PROFILE_FEATURES_PATH.name,
    "canonical_modeling_unit": "recurrence_occurrence",
    "occurrence_key": ["recurrence_key", "execution_start_time_us"],
    "occurrence_id_field": "occurrence_id",
    "execution_id_field": "execution_id",
    "recurrence_key_field": "recurrence_key",
    "timestamp_field": "execution_start_time_us",
    "split": {
        "type": "chronological_timestamp_group_preserving",
        "requested_train_fraction": 0.70,
        "requested_validation_fraction": 0.15,
        "requested_test_fraction": 0.15,
        "actual_train_execution_fraction": (
            len(train_exec_raw) / len(execution_features)
        ),
        "actual_validation_execution_fraction": (
            len(val_exec_raw) / len(execution_features)
        ),
        "actual_test_execution_fraction": (
            len(test_exec_raw) / len(execution_features)
        ),
    },
    "split_boundaries": split_metadata,
    "history_rule": (
        "For each occurrence, history contains only earlier occurrences of "
        "the same recurrence_key whose occurrence_end_time_us is <= the "
        "current execution_start_time_us. Overlapping/incomplete occurrences "
        "are excluded."
    ),
    "min_runs_candidates": MIN_RUNS_CANDIDATES,
    "min_runs_count_basis": (
        "Resource-specific counts of valid completed recurrence occurrences: "
        "occ_hist_peak_cpu_count and occ_hist_peak_memory_count. Execution rows "
        "do not count as separate runs."
    ),
    "selected_min_runs": SELECTED_MIN_RUNS,
    "selected_baseline_policy": SELECTED_BASELINE_POLICY,
    "baseline_selection_scope": "validation_only",
    "baseline_selection_objective": (
        "Lexicographic violation-focused ranking: any-resource violation %, "
        "worst single-resource violation %, CPU+memory violation sum, CPU waste, "
        "memory waste, then lower min_runs. Request is a comparator, not a "
        "historical-policy selection candidate."
    ),
    "target_policy_version": "next_distinct_occurrence_peak_v2",
    "target_definition": (
        "Within each frozen split and recurrence_key, target CPU/memory are the "
        "occurrence-level maximum peaks of the next distinct start-time occurrence."
    ),
    "prediction_time_convention": (
        "Features/history are those available at the start of the current "
        "occurrence; the target is the next occurrence. Request baseline carries "
        "forward the current occurrence request."
    ),
    "target_cpu_field": TARGET_CPU_COL,
    "target_memory_field": TARGET_MEMORY_COL,
    "target_timestamp_field": TARGET_TIME_COL,
    "target_split_isolation": (
        "Targets are constructed independently inside train, validation, and test; "
        "labels never cross split boundaries."
    ),
    "common_validation_population": {
        "required_fields": COMMON_EVAL_REQUIRED,
        "occurrences": len(validation_eval_occurrence_ids),
        "rule": (
            "All Request/Mean/P90/P95/P99 metrics and every min_runs candidate "
            "use the same occurrence IDs."
        ),
    },
    "test_policy": (
        "Test targets and deterministic recommendation columns may be materialized "
        "after validation selection, but N04 does not score test policies or use "
        "test performance for any decision."
    ),
    "canonical_occurrence_artifacts": {
        "train": TRAIN_OCCURRENCES_PATH.name,
        "validation": VAL_OCCURRENCES_PATH.name,
        "test": TEST_OCCURRENCES_PATH.name,
    },
    "compatibility_execution_artifacts": {
        "train": TRAIN_FEATURES_PATH.name,
        "validation": VAL_FEATURES_PATH.name,
        "test": TEST_FEATURES_PATH.name,
    },
    "validation_comparison_artifact": VALIDATION_COMPARISON_PATH.name,
}

metadata["selected_validation_metrics"] = (
    selected_validation_result.iloc[0].to_dict()
)

metadata


{'dataset': 'google_subset_v1',
 'source_artifact': 'job_metrics_v0.parquet',
 'canonical_modeling_unit': 'recurrence_occurrence',
 'occurrence_key': ['recurrence_key', 'execution_start_time_us'],
 'occurrence_id_field': 'occurrence_id',
 'execution_id_field': 'execution_id',
 'recurrence_key_field': 'recurrence_key',
 'timestamp_field': 'execution_start_time_us',
 'split': {'type': 'chronological_timestamp_group_preserving',
  'requested_train_fraction': 0.7,
  'requested_validation_fraction': 0.15,
  'requested_test_fraction': 0.15,
  'actual_train_execution_fraction': 0.7053177435573871,
  'actual_validation_execution_fraction': 0.144736478892679,
  'actual_test_execution_fraction': 0.14994577754993393},
 'split_boundaries': {'split_type': 'chronological_timestamp_group_preserving',
  'time_col': 'execution_start_time_us',
  'id_col': 'execution_id',
  'requested_train_fraction': 0.7,
  'requested_val_fraction': 0.15,
  'requested_test_fraction': 0.15000000000000005,
  'total_rows':

In [72]:
validation_comparison.to_csv(
    VALIDATION_COMPARISON_PATH,
    index=False,
)

# Canonical occurrence-level artifacts.
train_occurrences_final.to_parquet(TRAIN_OCCURRENCES_PATH, index=False)
val_occurrences_final.to_parquet(VAL_OCCURRENCES_PATH, index=False)
test_occurrences_final.to_parquet(TEST_OCCURRENCES_PATH, index=False)

# Compatibility execution-level artifacts.
train_features_final.to_parquet(TRAIN_FEATURES_PATH, index=False)
val_features_final.to_parquet(VAL_FEATURES_PATH, index=False)
test_features_final.to_parquet(TEST_FEATURES_PATH, index=False)

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        make_json_serializable(metadata),
        file,
        indent=2,
    )

print("Saved validation comparison:", VALIDATION_COMPARISON_PATH)
print("Saved canonical occurrence artifacts:")
print("  ", TRAIN_OCCURRENCES_PATH)
print("  ", VAL_OCCURRENCES_PATH)
print("  ", TEST_OCCURRENCES_PATH)
print("Saved compatibility execution artifacts:")
print("  ", TRAIN_FEATURES_PATH)
print("  ", VAL_FEATURES_PATH)
print("  ", TEST_FEATURES_PATH)
print("Saved metadata:", METADATA_PATH)


Saved validation comparison: /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/baseline_validation_comparison.csv
Saved canonical occurrence artifacts:
   /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/train_occurrences.parquet
   /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/val_occurrences.parquet
   /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/test_occurrences.parquet
Saved compatibility execution artifacts:
   /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/train_features.parquet
   /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/val_features.parquet
   /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/data/processed/google_subset_v1/test_features.parquet
Saved metadata: /Users/sonalilonkar/Desktop/TraceAdvisor/masters-project/da

## Saved Artifact Verification


In [73]:
saved_train_occ = pd.read_parquet(TRAIN_OCCURRENCES_PATH)
saved_val_occ = pd.read_parquet(VAL_OCCURRENCES_PATH)
saved_test_occ = pd.read_parquet(TEST_OCCURRENCES_PATH)

saved_train_exec = pd.read_parquet(TRAIN_FEATURES_PATH)
saved_val_exec = pd.read_parquet(VAL_FEATURES_PATH)
saved_test_exec = pd.read_parquet(TEST_FEATURES_PATH)

assert len(saved_train_occ) == len(train_occurrences_final)
assert len(saved_val_occ) == len(val_occurrences_final)
assert len(saved_test_occ) == len(test_occurrences_final)

assert len(saved_train_exec) == len(train_features_final)
assert len(saved_val_exec) == len(val_features_final)
assert len(saved_test_exec) == len(test_features_final)

required_occurrence_columns = [
    "occurrence_id",
    "prior_completed_occurrence_count",
    "occ_hist_peak_cpu_count",
    "occ_hist_peak_memory_count",
    TARGET_CPU_COL,
    TARGET_MEMORY_COL,
    TARGET_TIME_COL,
    "rec_cpu_request",
    "rec_cpu_mean",
    "rec_cpu_p90",
    "rec_cpu_p95",
    "rec_cpu_p99",
    "rec_memory_request",
    "rec_memory_mean",
    "rec_memory_p90",
    "rec_memory_p95",
    "rec_memory_p99",
]

for saved_df, split_name in [
    (saved_train_occ, "train"),
    (saved_val_occ, "validation"),
    (saved_test_occ, "test"),
]:
    missing = [
        column for column in required_occurrence_columns
        if column not in saved_df.columns
    ]
    if missing:
        raise RuntimeError(
            f"{split_name} occurrence artifact is missing: {missing}"
        )
    if not saved_df["occurrence_id"].is_unique:
        raise RuntimeError(
            f"{split_name} occurrence artifact contains duplicate occurrence_id."
        )

for saved_df, split_name in [
    (saved_train_exec, "train"),
    (saved_val_exec, "validation"),
    (saved_test_exec, "test"),
]:
    if saved_df["occurrence_id"].isna().any():
        raise RuntimeError(
            f"{split_name} execution artifact has unmapped occurrence rows."
        )
        
# ------------------------------------------------------------
# Verify validation comparison CSV
# ------------------------------------------------------------

saved_validation_comparison = pd.read_csv(
    VALIDATION_COMPARISON_PATH
)

assert len(saved_validation_comparison) == len(
    validation_comparison
)

assert set(saved_validation_comparison["policy"]) == set(
    validation_comparison["policy"]
)

assert set(saved_validation_comparison["min_runs"]) == set(
    validation_comparison["min_runs"]
)


# ------------------------------------------------------------
# Verify metadata JSON
# ------------------------------------------------------------

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    saved_metadata = json.load(file)

assert (
    saved_metadata["selected_min_runs"]
    == SELECTED_MIN_RUNS
)

assert (
    saved_metadata["selected_baseline_policy"]
    == SELECTED_BASELINE_POLICY
)

assert (
    saved_metadata["target_cpu_field"]
    == TARGET_CPU_COL
)

assert (
    saved_metadata["target_memory_field"]
    == TARGET_MEMORY_COL
)

assert (
    saved_metadata["baseline_selection_scope"]
    == "validation_only"
)

print("Saved artifact and metadata verification passed.")


Saved artifact and metadata verification passed.


# N04 Handoff

After this notebook runs successfully:

- the 70/15/15 chronological boundaries are frozen and timestamp groups do not cross boundaries;
- simultaneous execution rows are collapsed into one canonical occurrence for modeling/evaluation;
- baseline history uses only prior **completed occurrences**, never execution-row counts;
- CPU and memory `min_runs` eligibility uses resource-specific valid completed-occurrence counts;
- Request/Mean/P90/P95/P99 all use the same **next-occurrence** target definition;
- all validation configurations use exactly the same occurrence denominator;
- the selected policy and `min_runs` come from the corrected validation run rather than the old hard-coded P99/5 result;
- the test artifact is materialized only after selection and is not scored;
- `train_occurrences.parquet`, `val_occurrences.parquet`, and `test_occurrences.parquet` are the canonical N05/N06 modeling/evaluation artifacts;
- `train_features.parquet`, `val_features.parquet`, and `test_features.parquet` are compatibility execution-row artifacts with occurrence annotations merged back.

**Do not reintroduce `prior_execution_count`.** The canonical count is `prior_completed_occurrence_count`, and baseline eligibility uses `occ_hist_peak_cpu_count` / `occ_hist_peak_memory_count`.
